# Hansen Ch.9 Hypothesis Testing — 计算

**Chapter 9 Hypothesis Testing**

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch09_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：**Exercise 9.25–9.29**（Invest1993 / Nerlove / MRW / CPS 的 Wald 检验）。

> **写给只学过李子奈/陈强的同学：** 本章把检验建立在第 7 章渐近正态性之上。核心是 **Wald 统计量** = "估计值偏离零假设几个标准误"：
> $$W=(\hat\theta-\theta_0)'\hat V_{\hat\theta}^{-1}(\hat\theta-\theta_0)\xrightarrow{H_0}\chi^2_q.$$
> **三大检验**（trinity）：Wald（仅无约束）、LR（约束+无约束）、LM（仅约束），三者渐近等价（都 $\chi^2_q$），现代实证最常用 Wald。
> **CI–检验对偶**：拒绝 $H_0:\theta=\theta_0$（5%）⇔ $\theta_0$ 落在 95% CI 之外。
>
> **方差归一化（易错！）**：Wald 有两种等价写法——
> - 用**有限样本方差** $\hat V_{\hat\theta}=\hat R'\hat V_\beta\hat R$（=$\hat\beta$ 协方差矩阵估计经 delta）：$W=(\hat\theta-\theta_0)'\hat V_{\hat\theta}^{-1}(\hat\theta-\theta_0)$，**不带 $n$**。
> - 用**渐近方差** $\hat V_\theta$（使 $\sqrt n(\hat\theta-\theta)\to N(0,V_\theta)$）：$W=n(\hat\theta-\theta_0)'\hat V_\theta^{-1}(\hat\theta-\theta_0)$，**带 $n$**。
> 两者数值相同（$\hat V_{\hat\theta}=\hat V_\theta/n$）。**切勿混用**（既带 $n$ 又用有限样本方差会重复计 $n$）。下方 `wald()` 用 `R@V@R.T`（有限样本协方差矩阵），故**不带** $n$。


In [ ]:

import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def ols_hc(y, X, df_corr=True):
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    u = X * (e / np.clip(1 - h, 1e-12, None))[:, None]
    meat = u.T @ u
    if df_corr:
        meat = meat * (n / (n - k))
    V = XXinv @ meat @ XXinv
    return beta, e, V, n, k

def wald(R, beta, V, c=None):
    R = np.atleast_2d(R)
    if c is None:
        c = np.zeros(R.shape[0])
    d = R @ beta - c
    W = float(d.T @ np.linalg.inv(R @ V @ R.T) @ d)
    q = R.shape[0]
    return W, q, float(1 - stats.chi2.cdf(W, q))


## 9.25 Invest1993 (year=1987)

In [ ]:

inv = pd.read_excel(ROOT / "Invest1993/Invest1993.xlsx")
d = inv[inv.year == 1987][["inva", "vala", "cfa", "debta"]].apply(pd.to_numeric, errors="coerce").dropna()
y = d.inva.values
X = np.column_stack([d.vala, d.cfa, d.debta, np.ones(len(d))])
b, e, V, n, k = ols_hc(y, X)
print("n =", n)
print(pd.DataFrame({"beta": b, "SE": np.sqrt(np.diag(V)),
                    "lo": b - 1.96 * np.sqrt(np.diag(V)),
                    "hi": b + 1.96 * np.sqrt(np.diag(V))},
                   index=["Q", "C", "D", "int"]))
print("Wald C=D=0:", wald(np.array([[0,1,0,0],[0,0,1,0]], float), b, V))
print("Wald Q=0:", wald(np.array([[1,0,0,0]], float), b, V))
Q, C, D = d.vala.values, d.cfa.values, d.debta.values
Xq = np.column_stack([Q, C, D, Q**2, C**2, D**2, Q*C, Q*D, C*D, np.ones(len(d))])
bq, _, Vq, _, _ = ols_hc(y, Xq)
R6 = np.zeros((6, 10))
for i, j in enumerate([3, 4, 5, 6, 7, 8]):
    R6[i, j] = 1
print("Wald 6 nonlinear:", wald(R6, bq, Vq))


## 9.26 Nerlove1963

In [ ]:

ner = pd.read_excel(ROOT / "Nerlove1963/Nerlove1963.xlsx")
for c in ner.columns:
    ner[c] = pd.to_numeric(ner[c], errors="coerce")
ner = ner.dropna()
y = np.log(ner.Cost.values)
X = np.column_stack([np.ones(len(ner)), np.log(ner.output), np.log(ner.Plabor),
                     np.log(ner.Pcapital), np.log(ner.Pfuel)])
b, e, V, n, k = ols_hc(y, X)
print("n =", n)
print(pd.Series(b, index=["const", "logQ", "logPL", "logPK", "logPF"]))
print("SE:", np.sqrt(np.diag(V)))
print("Wald CRS:", wald(np.array([[0, 0, 1, 1, 1]], float), b, V, c=np.array([1.0])))


## 9.27 MRW1992

In [ ]:

mrw = pd.read_excel(ROOT / "MRW1992/MRW1992.xlsx")
m = mrw[mrw.N == 1]
y = (np.log(m.Y85) - np.log(m.Y60)).values
X = np.column_stack([
    np.log(m.Y60), np.log(m.invest / 100), np.log(m.pop_growth / 100 + 0.05),
    np.log(m.school / 100), np.ones(len(m))
])
b, e, V, n, k = ols_hc(y, X)
print("n =", n, "beta =", b)
print("Wald sum I+G+S = 0:", wald(np.array([[0, 1, 1, 1, 0]], float), b, V))


## 9.28–9.29 CPS marriage / education returns

In [ ]:

df = pd.read_excel(ROOT / "cps09mar/cps09mar.xlsx")
df["experience"] = df.age - df.education - 6
df["lwage"] = np.log(df.earnings / (df.hours * df.week))
df["exp2"] = (df.experience ** 2) / 100

black = df[(df.race == 2) & (df.hisp == 0)].copy().reset_index(drop=True)
for code, name in [(1, "m1"), (2, "m2"), (3, "m3"), (4, "wid"), (5, "div"), (6, "sep")]:
    black[name] = (black.marital == code).astype(float)
black["NE"] = (black.region == 1).astype(float)
black["South"] = (black.region == 3).astype(float)
black["West"] = (black.region == 4).astype(float)
y = black.lwage.to_numpy()
X = np.column_stack([
    black.education, black.experience, black.exp2, black.female,
    black.m1, black.m2, black.m3, black.wid, black.div, black.sep,
    black.NE, black.South, black.West, np.ones(len(black))
])
b, e, V, n, k = ols_hc(y, X)
R = np.zeros((6, 14))
for i, j in enumerate(range(4, 10)):
    R[i, j] = 1
print("9.28 n=", n, "Wald marriage=0:", wald(R, b, V))

sub = df[((df.race == 1) | (df.race == 2)) & (df.hisp == 0)].copy().reset_index(drop=True)
wm = ((sub.race == 1) & (sub.female == 0)).to_numpy().astype(float)
wf = ((sub.race == 1) & (sub.female == 1)).to_numpy().astype(float)
bm = ((sub.race == 2) & (sub.female == 0)).to_numpy().astype(float)
bf = ((sub.race == 2) & (sub.female == 1)).to_numpy().astype(float)
edu, exp, exp2 = sub.education.to_numpy(float), sub.experience.to_numpy(float), sub.exp2.to_numpy(float)
y = sub.lwage.to_numpy(float)
X = np.column_stack([edu * wm, edu * wf, edu * bm, edu * bf, exp, exp2, wf, bm, bf, np.ones(len(sub))])
b, e, V, n, k = ols_hc(y, X)
print("9.29 n=", n, "edu returns", b[:4])
R = np.array([[1, -1, 0, 0, 0, 0, 0, 0, 0, 0],
              [1, 0, -1, 0, 0, 0, 0, 0, 0, 0],
              [1, 0, 0, -1, 0, 0, 0, 0, 0, 0]], float)
print("9.29 Wald common return:", wald(R, b, V))


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch09：调整 $R^2$ 与 $|t|>1$、两样本 Wald 的 $\chi^2$ size、delta method 方差、CI–检验对偶、小 Wald 值的含义。可独立运行。

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(9)

# Ex 9.1: 加入新回归元后调整 R² 上升 ⇔ 该变量 |t|>1
n = 200
X0 = rng.standard_normal((n, 3))
inc, reps = 0, 5000
for r in range(reps):
    X1 = np.c_[X0, np.ones(n)]
    Xnew = rng.standard_normal(n)
    X2 = np.c_[X0, Xnew, np.ones(n)]
    Y = rng.standard_normal(n)
    def adjR2(X):
        kk = X.shape[1]
        b = np.linalg.lstsq(X, Y, rcond=None)[0]
        e = Y - X @ b
        return 1 - (n-1)/(n-kk) * ((e @ e) / ((Y - Y.mean()) @ Y))
    b2 = np.linalg.lstsq(X2, Y, rcond=None)[0]
    e2 = Y - X2 @ b2
    s2 = (e2 @ e2) / (n - X2.shape[1])
    tnew = b2[-2] / np.sqrt(s2 * np.linalg.inv(X2.T @ X2)[-2, -2])
    if (abs(tnew) > 1) == (adjR2(X2) > adjR2(X1)):
        inc += 1
print(f"[9.1] 调整R²升 ⇔ |t|>1  吻合率={inc/reps:.4f}")

# Ex 9.2: 两独立样本 Wald → χ²_k（H0: β₁=β₂ 为真时 size≈0.05）
# 【Wald 方差归一化】用“有限样本方差” V̂_j = σ̂_j² (X'X)^-1 时【不带额外 n 因子】，
#   因为 (X'X)^-1 本身已是 O(1/n)。若同时带 n 又用 (X'X)^-1，会重复计 n（见标题单元格警示）。
k, n, reps = 3, 100, 20000
beta = np.array([1.0, 0.5, -0.3]); rej = 0
for r in range(reps):
    X = rng.standard_normal((n, k))
    Y1 = X @ beta + rng.standard_normal(n)
    Y2 = X @ beta + rng.standard_normal(n)
    b1 = np.linalg.lstsq(X, Y1, rcond=None)[0]; e1 = Y1 - X @ b1
    b2 = np.linalg.lstsq(X, Y2, rcond=None)[0]; e2 = Y2 - X @ b2
    s21 = (e1 @ e1) / (n - k); s22 = (e2 @ e2) / (n - k)
    V1 = s21 * np.linalg.inv(X.T @ X)                            # β̂_j 的有限样本方差估计
    V2 = s22 * np.linalg.inv(X.T @ X)
    d = b2 - b1; Vd = V1 + V2                                    # 独立 ⇒ 方差相加
    W = float(d @ np.linalg.inv(Vd) @ d)                         # 不带 n（V̂_j 已含 1/n）
    if W > stats.chi2.ppf(0.95, k):
        rej += 1
print(f"[9.2] 两样本 Wald 的 size={rej/reps:.4f}  (应≈0.05)")

# Ex 9.10: delta method var(σ̂) = σ²/(2n)
sigma, nn = 2.0, 1000
sh = [np.sqrt((lambda e: (e @ e) / nn)(rng.standard_normal(nn) * sigma)) for _ in range(40000)]
print(f"[9.10] var(σ̂) MC={np.var(sh):.2e} ≈ σ²/(2n)={sigma**2/(2*nn):.2e}")

# Ex 9.14: CI–检验对偶（拒绝 H₀:θ=0 ⇔ 0 落在 95% CI 之外）
n, agree = 100, 0
for r in range(10000):
    X = rng.standard_normal(n); Y = 0.3 * X + rng.standard_normal(n)
    b = np.sum(X*Y) / np.sum(X**2)
    se = np.sqrt(np.sum((Y - X*b)**2) / (n-1) / np.sum(X**2))
    if (abs(b/se) > 1.96) == (not (b - 1.96*se <= 0 <= b + 1.96*se)):
        agree += 1
print(f"[9.14] 拒绝(T) ⇔ θ₀∉CI  吻合率={agree/10000:.4f}")
print(f"[9.19] χ²₁ 的 5% 临界值={stats.chi2.ppf(0.95, 1):.3f}; W=0.34 < 临界 ⇒ 不拒绝(小 W 支持 H₀)")